# BERT 미세조정 — Disaster Tweets (PyTorch + transformers) — Colab

**BERT**(Transformer 기반 사전학습 언어모델)로 [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) (재난 트윗 분류) 를 **미세조정(fine-tuning)** 하는 기본 예제입니다.

- 사전학습된 `bert-base-uncased` 에 분류 헤드를 붙여 2클래스(재난/일반)로 학습합니다.
- 핵심 흐름: **토크나이저 → BERT 인코딩 → [CLS] 분류 → 미세조정**.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` (`id,target`) 가 `/content` 에 생성됩니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (BERT 학습은 CPU에서 매우 느림).  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> 💡 같은 대회의 07번(TF-IDF+Ridge) 노트북과 비교하면 전통 ML vs Transformer 차이를 볼 수 있습니다.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "transformers", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Disaster Tweets 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 로드 & 토크나이즈

BERT 토크나이저로 트윗 텍스트를 토큰 id 로 변환합니다 (최대 길이 128, 패딩).

In [ ]:
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
print("train:", train.shape, "| test:", test.shape)

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(list(texts), truncation=True, padding="max_length",
                     max_length=MAX_LEN, return_tensors="pt")

enc_train = encode(train["text"].fillna(""))
enc_test = encode(test["text"].fillna(""))
labels = torch.tensor(train["target"].values, dtype=torch.long)
print("input_ids:", tuple(enc_train["input_ids"].shape))

## 3. Dataset / DataLoader & train/valid 분할

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

full = TensorDataset(enc_train["input_ids"], enc_train["attention_mask"], labels)
n_val = int(len(full) * 0.1); n_tr = len(full) - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full, [n_tr, n_val], generator=g)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| train:", n_tr, "valid:", n_val)

## 4. BERT 분류 모델 불러오기

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
print("모델 로드 완료:", MODEL_NAME)

## 5. 미세조정 학습

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 2

for epoch in range(1, EPOCHS + 1):
    model.train()
    for ids, mask, y in train_loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=y)
        out.loss.backward(); optimizer.step()
    # 검증
    model.eval(); preds=[]; gts=[]
    with torch.no_grad():
        for ids, mask, y in val_loader:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            preds.extend(logits.argmax(1).cpu().numpy()); gts.extend(y.numpy())
    print(f"epoch {epoch} | val acc {accuracy_score(gts,preds):.4f} | val f1 {f1_score(gts,preds):.4f}")

## 6. 테스트 예측
학습된 모델로 테스트셋을 예측합니다.

In [ ]:
test_loader = DataLoader(TensorDataset(enc_test["input_ids"], enc_test["attention_mask"]), batch_size=32)

model.eval(); test_preds=[]
with torch.no_grad():
    for ids, mask in test_loader:
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        test_preds.extend(logits.argmax(1).cpu().numpy())

## 7. 제출 파일 생성
예측을 `submission.csv`(id, target)로 저장합니다.

In [ ]:
submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = pd.DataFrame({"id": test["id"], "target": np.array(test_preds, dtype=int)})
sub.to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(sub))
sub.head()

## 8. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[NLP Getting Started 제출 페이지](https://www.kaggle.com/c/nlp-getting-started/submit)**

- 대회 홈: https://www.kaggle.com/c/nlp-getting-started
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.